# HDF5 Data Inspection Notebook
## Purpose: Inspect Factor and Label Datasets for Training Factor Reweight Models

This notebook provides a detailed inspection of:
1. **weakFactors.h5** - Factor values in pandas fixed format (use h5py only!)
2. **Label10.h5** - Label data in pandas table format (safe for pd.read_hdf)

### Critical Notes:
- ❌ **NEVER** use `pd.read_hdf(..., start=, stop=)` on fixed-format files
- ✅ Use `h5py` for safe slicing of weakFactors.h5
- ✅ Use `pd.read_hdf(..., where=...)` for Label10.h5

## 1. Import Required Libraries

In [ ]:
import h5py
import pandas as pd
import numpy as np
from pathlib import Path

# Configure paths - UPDATE THESE TO YOUR ACTUAL FILE LOCATIONS
FACTOR_FILE = Path("weakFactors.h5")
LABEL_FILE = Path("Label10.h5")

print("Libraries imported successfully!")
print(f"Factor file exists: {FACTOR_FILE.exists()}")
print(f"Label file exists: {LABEL_FILE.exists()}")

## 2. Inspect weakFactors.h5 Structure with h5py

Open the factor file and recursively list all groups and datasets to understand the internal layout.
Each key (/0, /1, /2) contains one factor matrix with identical structure.

In [ ]:
def print_hdf5_structure(name, obj):
    """Recursively print HDF5 structure with shapes and dtypes"""
    indent = "  " * name.count("/")
    if isinstance(obj, h5py.Dataset):
        print(f"{indent}📊 {name}: shape={obj.shape}, dtype={obj.dtype}")
    else:
        print(f"{indent}📁 {name}/")

# Open factor file and explore structure
with h5py.File(FACTOR_FILE, "r") as f:
    print("=" * 60)
    print("weakFactors.h5 Structure (FIXED format)")
    print("=" * 60)
    print(f"\nTop-level keys: {list(f.keys())}")
    print("\nFull structure:")
    f.visititems(print_hdf5_structure)

## 3. Examine Factor Matrix Metadata and Shapes

For each key (/0, /1, /2), inspect the core datasets:
- `block0_values`: The actual factor values (2,819,791 × 1,128)
- `axis0`: Column labels (factor IDs)
- `axis1_label0/1`: Index codes for MultiIndex reconstruction
- `axis1_level0/1`: Unique values for each index level

In [ ]:
with h5py.File(FACTOR_FILE, "r") as f:
    for key in ["0", "1", "2"]:
        print(f"\n{'='*60}")
        print(f"Factor Matrix: /{key}")
        print("="*60)
        
        grp = f[key]
        
        # Core datasets to inspect
        datasets = [
            "block0_values",  # Actual factor values
            "axis0",          # Column labels (factor IDs)
            "block0_items",   # Same as axis0
            "axis1_label0",   # Index level 0 codes
            "axis1_label1",   # Index level 1 codes
            "axis1_level0",   # Unique dates
            "axis1_level1",   # Unique instruments
        ]
        
        for ds_name in datasets:
            if ds_name in grp:
                ds = grp[ds_name]
                print(f"  {ds_name:20s} shape={str(ds.shape):20s} dtype={ds.dtype}")
        
        # Memory estimate
        values_shape = grp["block0_values"].shape
        mem_gb = np.prod(values_shape) * 4 / (1024**3)  # float32 = 4 bytes
        print(f"\n  Memory per matrix: ~{mem_gb:.2f} GB")

## 4. Reconstruct MultiIndex from Raw Components

The MultiIndex is stored as coded arrays:
- `axis1_level0`: Unique trading dates (933 values)
- `axis1_level1`: Unique instrument IDs (3646 values)  
- `axis1_label0[i]`: Index into level0 for row i
- `axis1_label1[i]`: Index into level1 for row i

**Formula**: `index[i] = (level0[label0[i]], level1[label1[i]])`

In [ ]:
def load_index_components(h5_file, key="0"):
    """Load all components needed to reconstruct the MultiIndex"""
    with h5py.File(h5_file, "r") as f:
        grp = f[key]
        
        # Load level arrays (small, load fully)
        level0 = grp["axis1_level0"][:]  # Dates (int64)
        level1 = grp["axis1_level1"][:]  # Instruments (bytes)
        
        # Decode bytes to strings if needed
        if level1.dtype.kind == 'S':  # byte string
            level1 = np.array([x.decode('utf-8') if isinstance(x, bytes) else x for x in level1])
        
        # Load label arrays (codes - int16)
        label0 = grp["axis1_label0"][:]
        label1 = grp["axis1_label1"][:]
        
        # Load column names
        columns = grp["axis0"][:]
        
    return level0, level1, label0, label1, columns

# Load index components for key /0
level0, level1, label0, label1, columns = load_index_components(FACTOR_FILE, "0")

print("Index Components Loaded:")
print(f"  level0 (dates):       {level0.shape}, dtype={level0.dtype}")
print(f"  level1 (instruments): {level1.shape}, dtype={level1.dtype}")
print(f"  label0 (date codes):  {label0.shape}, dtype={label0.dtype}")
print(f"  label1 (inst codes):  {label1.shape}, dtype={label1.dtype}")
print(f"  columns (factors):    {columns.shape}, dtype={columns.dtype}")
print(f"\nSample dates (first 5):      {level0[:5]}")
print(f"Sample instruments (first 5): {level1[:5]}")

In [ ]:
def reconstruct_multiindex(level0, level1, label0, label1, start=0, stop=None):
    """
    Reconstruct pandas MultiIndex for a slice of rows.
    
    Parameters:
    -----------
    level0, level1: Full unique value arrays
    label0, label1: Full or sliced code arrays
    start, stop: Row slice range
    """
    # Slice the codes
    codes0 = label0[start:stop]
    codes1 = label1[start:stop]
    
    # Map codes to actual values
    dates = level0[codes0]
    instruments = level1[codes1]
    
    # Create MultiIndex
    index = pd.MultiIndex.from_arrays(
        [dates, instruments],
        names=["date", "instrument"]
    )
    return index

# Reconstruct index for first 10 rows
sample_index = reconstruct_multiindex(level0, level1, label0, label1, start=0, stop=10)
print("Reconstructed MultiIndex (first 10 rows):")
print(sample_index)
print(f"\nIndex levels: {sample_index.names}")
print(f"Index dtype: level0={sample_index.get_level_values(0).dtype}, level1={sample_index.get_level_values(1).dtype}")

## 5. Sample Factor Values Using Safe Slicing

Use h5py to slice `block0_values[start:stop, :]` and create a proper DataFrame.

⚠️ **Remember**: Never use `pd.read_hdf(..., start=, stop=)` on this file!

In [ ]:
def load_factor_slice(h5_file, key="0", start=0, stop=1000):
    """
    Safely load a slice of factor data using h5py.
    
    Returns a pandas DataFrame with proper MultiIndex.
    """
    with h5py.File(h5_file, "r") as f:
        grp = f[key]
        
        # Load factor values slice
        values = grp["block0_values"][start:stop, :]
        
        # Load columns
        columns = grp["axis0"][:]
        
        # Load index components
        level0 = grp["axis1_level0"][:]
        level1 = grp["axis1_level1"][:]
        label0 = grp["axis1_label0"][start:stop]
        label1 = grp["axis1_label1"][start:stop]
        
        # Decode bytes if needed
        if level1.dtype.kind == 'S':
            level1 = np.array([x.decode('utf-8') if isinstance(x, bytes) else x for x in level1])
    
    # Reconstruct index
    dates = level0[label0]
    instruments = level1[label1]
    index = pd.MultiIndex.from_arrays([dates, instruments], names=["date", "instrument"])
    
    # Create DataFrame
    df = pd.DataFrame(values, index=index, columns=columns)
    return df

# Load a sample slice
sample_df = load_factor_slice(FACTOR_FILE, key="0", start=0, stop=100)

print(f"Sample DataFrame shape: {sample_df.shape}")
print(f"Memory usage: {sample_df.memory_usage(deep=True).sum() / 1024:.2f} KB")
print(f"\nDataFrame info:")
print(f"  Index: {sample_df.index.names}")
print(f"  Columns: {len(sample_df.columns)} factors")
print(f"  Dtype: {sample_df.dtypes.unique()}")
print(f"\nFirst few rows:")
sample_df.head()

In [ ]:
# Check for NaN values and basic statistics
print("Factor Statistics (sample):")
print(f"  NaN count: {sample_df.isna().sum().sum()}")
print(f"  NaN ratio: {sample_df.isna().sum().sum() / sample_df.size:.4%}")
print(f"\nDescriptive stats (first 5 factors):")
sample_df.iloc[:, :5].describe()

## 6. Inspect Label10.h5 Table Structure

The label file uses pandas TABLE format which supports safe slicing with `where` clauses.
Let's explore its structure using h5py first.

In [ ]:
# Explore Label10.h5 structure
with h5py.File(LABEL_FILE, "r") as f:
    print("=" * 60)
    print("Label10.h5 Structure (TABLE format)")
    print("=" * 60)
    print(f"\nTop-level keys: {list(f.keys())}")
    print("\nFull structure:")
    f.visititems(print_hdf5_structure)

In [ ]:
# Inspect the index accelerator components
with h5py.File(LABEL_FILE, "r") as f:
    print("Index Accelerator Components (/Data/_i_table/index/):")
    print("-" * 50)
    
    if "Data" in f and "_i_table" in f["Data"]:
        idx_grp = f["Data/_i_table/index"]
        for name in idx_grp.keys():
            ds = idx_grp[name]
            print(f"  {name:15s} shape={str(ds.shape):20s} dtype={ds.dtype}")
    else:
        print("  Index group not found - structure may differ")

## 7. Query Label Data with Safe Filtering

Use `pd.read_hdf` with `where` clause to safely query the table-format label file.
This is safe because the file uses TABLE format with index acceleration.

In [ ]:
# First, let's check what keys are available in the label file
with pd.HDFStore(LABEL_FILE, mode="r") as store:
    print("Available keys in Label10.h5:")
    print(store.keys())
    
    # Get info about the main table
    print("\nTable info:")
    for key in store.keys():
        print(f"\n{key}:")
        try:
            info = store.get_storer(key)
            print(f"  Type: {type(info).__name__}")
            print(f"  nrows: {info.nrows}")
            if hasattr(info, 'data_columns'):
                print(f"  data_columns: {info.data_columns}")
        except Exception as e:
            print(f"  Error: {e}")

In [ ]:
# Safe query using start/stop (TABLE format supports this)
# Load a small sample first
label_sample = pd.read_hdf(LABEL_FILE, key="Data", start=0, stop=1000)

print(f"Label DataFrame shape: {label_sample.shape}")
print(f"Label columns: {list(label_sample.columns)}")
print(f"Label index: {label_sample.index.names}")
print(f"Label dtypes:")
print(label_sample.dtypes)
print(f"\nFirst few rows:")
label_sample.head(10)

In [ ]:
# Example: Query with WHERE clause (if data_columns are available)
# This demonstrates safe filtering for TABLE format

# First check the index structure
print("Label Index Structure:")
print(f"  Index type: {type(label_sample.index)}")
if isinstance(label_sample.index, pd.MultiIndex):
    print(f"  Index names: {label_sample.index.names}")
    print(f"  Index levels: {[len(level) for level in label_sample.index.levels]}")
else:
    print(f"  Index name: {label_sample.index.name}")
    print(f"  Index dtype: {label_sample.index.dtype}")

# Show unique dates in sample
if "date" in label_sample.index.names or "date" in label_sample.columns:
    if "date" in label_sample.index.names:
        dates = label_sample.index.get_level_values("date").unique()
    else:
        dates = label_sample["date"].unique()
    print(f"\nSample dates (first 5): {dates[:5]}")

## 8. Align Factor and Label Indices

For training factor reweight models, we need to align:
- Factor data from `weakFactors.h5` (date × instrument → factor values)
- Label data from `Label10.h5` (date × instrument → target labels)

Both should share the same (date, instrument) index.

In [ ]:
# Analyze the structural difference between factor and label indices

print("=" * 60)
print("STRUCTURAL COMPARISON")
print("=" * 60)

# Factor structure
print("\n📊 FACTOR DATA:")
print(f"  Index type: MultiIndex (date, instrument)")
print(f"  Unique dates: {len(level0)} → range: {level0.min()} to {level0.max()}")
print(f"  Unique instruments: {len(level1)}")
print(f"  Total rows: {len(label0):,}")
print(f"  Date format: int64 (e.g., {level0[0]})")
print(f"  Instrument format: string (e.g., '{level1[0]}')")

# Label structure  
print("\n📊 LABEL DATA:")
print(f"  Index: endDate (single index, int64)")
print(f"  Columns: {list(label_sample.columns)}")
print(f"  Total rows in file: 12,519,451")
print(f"  Date format: int64 (e.g., {label_sample.index[0]})")
print(f"  Instrument in column 'code': (e.g., '{label_sample['code'].iloc[0]}')")

# Key insight
print("\n" + "=" * 60)
print("🔑 KEY INSIGHT: Different index structures!")
print("=" * 60)
print("""
  Factor:  MultiIndex(date, instrument) 
  Label:   Index(endDate) + column 'code' for instrument
  
  To align:
    - Factor 'date' ↔ Label 'endDate' (index)
    - Factor 'instrument' ↔ Label 'code' (column)
""")

In [ ]:
# The label file is sorted chronologically - first 100k rows are from 2006-2007
# Factor data is from 2018-2021, so we need to query the right date range

# Key rules:
# 1. labelDate must align with factor date
# 2. endDate > 20211231 cannot be used (future data filter)

print("=" * 60)
print("LABEL FIELD RULES")
print("=" * 60)
print("""
📋 Label Fields:
  - labelDate: Must align with factor 'date' 
  - endDate: Used to filter future data
  
⚠️ Filter Rule: endDate > 20211231 cannot be used
""")

# Check if we can query by date using WHERE clause
with pd.HDFStore(LABEL_FILE, mode="r") as store:
    storer = store.get_storer("Data")
    print(f"Data columns available for WHERE: {storer.data_columns}")
    
# Since data_columns is empty, we need to scan to find right rows
# Let's estimate where 2018+ data starts

# Load from middle/end of file to find 2018+ data
total_rows = 12519451
test_positions = [
    (0, 1000, "Start"),
    (total_rows // 2, total_rows // 2 + 1000, "Middle"),
    (total_rows - 10000, total_rows, "End"),
]

print("\nScanning label file for date ranges:")
print("-" * 50)
for start, stop, pos in test_positions:
    sample = pd.read_hdf(LABEL_FILE, key="Data", start=start, stop=stop)
    sample = sample.reset_index()
    print(f"{pos:8s} (rows {start:,}-{stop:,}): labelDate {sample['labelDate'].min()}-{sample['labelDate'].max()}, endDate {sample['endDate'].min()}-{sample['endDate'].max()}")

## 9. Find Labels Matching Factor Date Range

Since the label file is sorted chronologically and covers 2006-2021+, we need to:
1. **Binary search** to find where `labelDate >= 20180102` starts
2. **Filter out** rows where `endDate > 20211231` (future data)
3. **Align** on `(labelDate, code)` ↔ `(date, instrument)`

In [ ]:
# Binary search to find where labelDate >= 20180102 starts

def find_label_date_position(label_file, target_date, total_rows):
    """Binary search to find approximate row where labelDate >= target_date"""
    low, high = 0, total_rows
    
    while low < high:
        mid = (low + high) // 2
        sample = pd.read_hdf(label_file, key="Data", start=mid, stop=mid+1)
        sample = sample.reset_index()
        mid_date = sample['labelDate'].iloc[0]
        
        if mid_date < target_date:
            low = mid + 1
        else:
            high = mid
    
    return low

# Find start position for 2018 data
FACTOR_START_DATE = 20180102
FACTOR_END_DATE = 20211230
FUTURE_CUTOFF = 20211231  # endDate > this cannot be used

print("Searching for label rows matching factor date range...")
print(f"Target: labelDate >= {FACTOR_START_DATE}")

total_label_rows = 12519451
start_pos = find_label_date_position(LABEL_FILE, FACTOR_START_DATE, total_label_rows)

print(f"\n✅ Found: labelDate >= {FACTOR_START_DATE} starts at row ~{start_pos:,}")

# Verify
verify_sample = pd.read_hdf(LABEL_FILE, key="Data", start=max(0, start_pos-5), stop=start_pos+5)
verify_sample = verify_sample.reset_index()
print(f"\nVerification (rows around position {start_pos}):")
print(verify_sample[['endDate', 'labelDate', 'code']].to_string())

In [ ]:
# Load labels for factor date range and apply future data filter

def load_valid_labels(label_file, start_row, n_rows=500000, future_cutoff=20211231):
    """
    Load labels and filter:
    1. labelDate aligns with factor date
    2. endDate <= future_cutoff (no future data)
    """
    labels = pd.read_hdf(label_file, key="Data", start=start_row, stop=start_row + n_rows)
    labels = labels.reset_index()
    
    print(f"Loaded {len(labels):,} label rows")
    print(f"  labelDate range: {labels['labelDate'].min()} to {labels['labelDate'].max()}")
    print(f"  endDate range: {labels['endDate'].min()} to {labels['endDate'].max()}")
    
    # Filter: endDate <= future_cutoff
    before_filter = len(labels)
    labels = labels[labels['endDate'] <= future_cutoff]
    after_filter = len(labels)
    
    print(f"\n⚠️ Future data filter (endDate > {future_cutoff}):")
    print(f"  Removed: {before_filter - after_filter:,} rows")
    print(f"  Remaining: {after_filter:,} rows")
    
    # Set MultiIndex for alignment: (labelDate, code) -> (date, instrument)
    labels = labels.set_index(['labelDate', 'code'])
    labels.index.names = ['date', 'instrument']
    
    return labels

# Load valid labels
labels_valid = load_valid_labels(LABEL_FILE, start_row=start_pos, n_rows=500000, future_cutoff=FUTURE_CUTOFF)
print(f"\nValid labels shape: {labels_valid.shape}")
print(f"Columns: {list(labels_valid.columns)}")
labels_valid.head()

## 10. Create Aligned Training Dataset

In [ ]:
# Now align factors with valid labels

# Load a batch of factors
factors = load_factor_slice(FACTOR_FILE, key="0", start=0, stop=50000)

print("ALIGNMENT:")
print("=" * 60)
print(f"Factor shape: {factors.shape}")
print(f"Factor index sample: {factors.index[:3].tolist()}")
print(f"Factor date range: {factors.index.get_level_values('date').min()} to {factors.index.get_level_values('date').max()}")

print(f"\nLabel shape: {labels_valid.shape}")  
print(f"Label index sample: {labels_valid.index[:3].tolist()}")
print(f"Label date range: {labels_valid.index.get_level_values('date').min()} to {labels_valid.index.get_level_values('date').max()}")

# Find intersection
common_idx = factors.index.intersection(labels_valid.index)
print(f"\n✅ Common index (aligned samples): {len(common_idx):,}")

if len(common_idx) > 0:
    X = factors.loc[common_idx]
    y = labels_valid.loc[common_idx, 'labelValue']
    
    print(f"\n📊 ALIGNED TRAINING DATA:")
    print(f"  X shape: {X.shape} (samples × factors)")
    print(f"  y shape: {y.shape}")
else:
    print("\n⚠️ No alignment - checking index format...")
    print(f"Factor instrument type: {type(factors.index.get_level_values('instrument')[0])}")
    print(f"Label instrument type: {type(labels_valid.index.get_level_values('instrument')[0])}")

## 11. Validate Aligned Data

In [ ]:
# Validate aligned training data

if len(common_idx) > 0:
    print("=" * 60)
    print("✅ TRAINING DATA VALIDATION")
    print("=" * 60)
    
    print(f"\n📊 Features (X):")
    print(f"  Shape: {X.shape}")
    print(f"  Dtype: {X.dtypes.unique()}")
    print(f"  NaN ratio: {X.isna().sum().sum() / X.size:.4%}")
    print(f"  Value range: [{X.values.min():.4f}, {X.values.max():.4f}]")
    
    print(f"\n🎯 Target (y):")
    print(f"  Shape: {y.shape}")
    print(f"  Dtype: {y.dtype}")
    print(f"  NaN count: {y.isna().sum()}")
    print(f"  Value range: [{y.min():.6f}, {y.max():.6f}]")
    print(f"  Mean: {y.mean():.6f}, Std: {y.std():.6f}")
    
    print(f"\n📅 Coverage:")
    dates_covered = X.index.get_level_values('date').unique()
    instruments_covered = X.index.get_level_values('instrument').unique()
    print(f"  Dates: {len(dates_covered)} unique")
    print(f"  Date range: {dates_covered.min()} to {dates_covered.max()}")
    print(f"  Instruments: {len(instruments_covered)} unique")
    
    print(f"\n📋 Sample data (first 5 rows):")
    sample_aligned = pd.concat([X.iloc[:5, :3], y.iloc[:5]], axis=1)
    display(sample_aligned)
else:
    print("❌ Alignment failed - see troubleshooting above")

## 12. Complete Training Data Loader

In [ ]:
class FactorLabelTrainingData:
    """
    Complete training data loader with proper alignment rules.
    
    Rules:
    - labelDate must align with factor date
    - endDate > 20211231 cannot be used (future data filter)
    """
    
    def __init__(self, factor_file, label_file, factor_key="0", future_cutoff=20211231):
        self.factor_file = Path(factor_file)
        self.label_file = Path(label_file)
        self.factor_key = factor_key
        self.future_cutoff = future_cutoff
        
        self._load_metadata()
    
    def _load_metadata(self):
        """Load factor metadata"""
        with h5py.File(self.factor_file, "r") as f:
            grp = f[self.factor_key]
            self.n_factor_rows = grp["block0_values"].shape[0]
            self.n_factors = grp["block0_values"].shape[1]
            self.factor_columns = grp["axis0"][:]
            self.level0 = grp["axis1_level0"][:]  # dates
            self.level1 = grp["axis1_level1"][:]  # instruments
            
            if self.level1.dtype.kind == 'S':
                self.level1 = np.array([x.decode('utf-8') for x in self.level1])
        
        self.factor_date_range = (self.level0.min(), self.level0.max())
        print(f"Factor: {self.n_factor_rows:,} rows × {self.n_factors} factors")
        print(f"Date range: {self.factor_date_range[0]} to {self.factor_date_range[1]}")
    
    def load_labels_for_date_range(self, start_date, end_date, label_start_row=0):
        """Load and filter labels for a date range"""
        # Load labels in chunks to find matching date range
        chunk_size = 500000
        all_labels = []
        
        with pd.HDFStore(self.label_file, mode='r') as store:
            total_rows = store.get_storer('Data').nrows
        
        for start in range(label_start_row, total_rows, chunk_size):
            labels = pd.read_hdf(self.label_file, key="Data", start=start, stop=start+chunk_size)
            labels = labels.reset_index()
            
            # Filter by labelDate range
            labels = labels[(labels['labelDate'] >= start_date) & (labels['labelDate'] <= end_date)]
            
            # Filter: endDate <= future_cutoff
            labels = labels[labels['endDate'] <= self.future_cutoff]
            
            if len(labels) > 0:
                all_labels.append(labels)
            
            # Stop if we've passed the end date
            if labels['labelDate'].max() > end_date if len(labels) > 0 else False:
                break
        
        if all_labels:
            labels_combined = pd.concat(all_labels, ignore_index=True)
            labels_combined = labels_combined.set_index(['labelDate', 'code'])
            labels_combined.index.names = ['date', 'instrument']
            return labels_combined
        return None
    
    def create_training_data(self, factor_start=0, factor_stop=None):
        """Create aligned (X, y) training data"""
        if factor_stop is None:
            factor_stop = self.n_factor_rows
        
        # Load factors
        X = load_factor_slice(self.factor_file, self.factor_key, factor_start, factor_stop)
        
        # Get date range from factors
        dates = X.index.get_level_values('date')
        start_date, end_date = dates.min(), dates.max()
        
        # Load matching labels
        labels = self.load_labels_for_date_range(start_date, end_date)
        
        if labels is None:
            return None, None
        
        # Align
        common_idx = X.index.intersection(labels.index)
        X_aligned = X.loc[common_idx]
        y_aligned = labels.loc[common_idx, 'labelValue']
        
        return X_aligned, y_aligned

print("FactorLabelTrainingData class defined ✅")
print("\nUsage:")
print("  loader = FactorLabelTrainingData(FACTOR_FILE, LABEL_FILE)")
print("  X, y = loader.create_training_data(factor_start=0, factor_stop=100000)")

## 13. Export All Aligned Data for Training

Export all factor data (keys 0, 1, 2) with aligned labels for the date range 20180102-20211230.

**Output format**: Parquet (fast, compressed, columnar storage)

**Output files**:
- `factors_0.parquet` - Factor matrix 0 with aligned labels
- `factors_1.parquet` - Factor matrix 1 with aligned labels  
- `factors_2.parquet` - Factor matrix 2 with aligned labels
- `labels_aligned.parquet` - All valid labels

In [ ]:
# Configuration
OUTPUT_DIR = Path("processed_data")
OUTPUT_DIR.mkdir(exist_ok=True)

FACTOR_DATE_START = 20180102
FACTOR_DATE_END = 20211230
FUTURE_CUTOFF = 20211231

print("=" * 60)
print("EXPORT CONFIGURATION")
print("=" * 60)
print(f"Factor date range: {FACTOR_DATE_START} to {FACTOR_DATE_END}")
print(f"Future cutoff: endDate > {FUTURE_CUTOFF} excluded")
print(f"Output directory: {OUTPUT_DIR.absolute()}")

In [ ]:
# Step 1: Load ALL valid labels for the date range

def load_all_valid_labels(label_file, start_date, end_date, future_cutoff, label_start_row=0):
    """
    Load all labels where:
    - labelDate >= start_date AND labelDate <= end_date
    - endDate <= future_cutoff
    """
    print("Loading all valid labels...")
    
    with pd.HDFStore(label_file, mode='r') as store:
        total_rows = store.get_storer('Data').nrows
    
    chunk_size = 500000
    all_labels = []
    rows_loaded = 0
    rows_kept = 0
    
    for start in range(label_start_row, total_rows, chunk_size):
        labels = pd.read_hdf(label_file, key="Data", start=start, stop=start + chunk_size)
        labels = labels.reset_index()
        rows_loaded += len(labels)
        
        # Check if we've passed the date range
        if labels['labelDate'].min() > end_date:
            print(f"  Reached end of date range at row {start:,}")
            break
        
        # Filter by labelDate range
        labels = labels[(labels['labelDate'] >= start_date) & (labels['labelDate'] <= end_date)]
        
        # Filter: endDate <= future_cutoff
        labels = labels[labels['endDate'] <= future_cutoff]
        
        if len(labels) > 0:
            all_labels.append(labels)
            rows_kept += len(labels)
        
        if start % 2000000 == 0:
            print(f"  Processed {start:,} rows, kept {rows_kept:,} labels...")
    
    print(f"\n✅ Total rows scanned: {rows_loaded:,}")
    print(f"✅ Valid labels kept: {rows_kept:,}")
    
    if all_labels:
        labels_combined = pd.concat(all_labels, ignore_index=True)
        return labels_combined
    return None

# Load all valid labels
all_labels = load_all_valid_labels(
    LABEL_FILE, 
    start_date=FACTOR_DATE_START, 
    end_date=FACTOR_DATE_END, 
    future_cutoff=FUTURE_CUTOFF,
    label_start_row=start_pos  # Start from where 2018 data begins
)

print(f"\nAll labels shape: {all_labels.shape}")
print(f"Columns: {list(all_labels.columns)}")
print(f"labelDate range: {all_labels['labelDate'].min()} to {all_labels['labelDate'].max()}")
print(f"endDate range: {all_labels['endDate'].min()} to {all_labels['endDate'].max()}")

In [ ]:
# Step 2: Create label lookup index for fast alignment

# Set MultiIndex: (labelDate, code) -> (date, instrument)
labels_indexed = all_labels.set_index(['labelDate', 'code'])
labels_indexed.index.names = ['date', 'instrument']

print(f"Labels indexed shape: {labels_indexed.shape}")
print(f"Unique dates: {labels_indexed.index.get_level_values('date').nunique()}")
print(f"Unique instruments: {labels_indexed.index.get_level_values('instrument').nunique()}")

# Save labels
labels_output_path = OUTPUT_DIR / "labels_aligned.parquet"
labels_indexed.to_parquet(labels_output_path)
print(f"\n✅ Labels saved to: {labels_output_path}")
print(f"   File size: {labels_output_path.stat().st_size / 1024 / 1024:.2f} MB")

In [ ]:
# Step 3: Export all factor data for keys 0, 1, 2

def export_factor_data(factor_file, key, labels_indexed, output_dir, chunk_size=100000):
    """
    Load all factor data for a key and save with aligned labels.
    
    Saves:
    - factors_{key}.parquet: Factor values with MultiIndex
    - factors_{key}_aligned.parquet: Factors + labelValue for rows with labels
    """
    print(f"\n{'='*60}")
    print(f"Processing Factor Key: /{key}")
    print("=" * 60)
    
    # Get total rows
    with h5py.File(factor_file, "r") as f:
        n_rows = f[key]["block0_values"].shape[0]
        n_cols = f[key]["block0_values"].shape[1]
    
    print(f"Total rows: {n_rows:,}, columns: {n_cols}")
    
    # Load in chunks and save
    all_chunks = []
    aligned_chunks = []
    
    for start in range(0, n_rows, chunk_size):
        stop = min(start + chunk_size, n_rows)
        
        # Load factor slice
        factors = load_factor_slice(factor_file, key=key, start=start, stop=stop)
        all_chunks.append(factors)
        
        # Align with labels
        common_idx = factors.index.intersection(labels_indexed.index)
        if len(common_idx) > 0:
            factors_aligned = factors.loc[common_idx].copy()
            factors_aligned['labelValue'] = labels_indexed.loc[common_idx, 'labelValue']
            factors_aligned['endDate'] = labels_indexed.loc[common_idx, 'endDate']
            aligned_chunks.append(factors_aligned)
        
        if start % 500000 == 0:
            print(f"  Processed {start:,}/{n_rows:,} rows...")
    
    # Combine all chunks
    print("  Combining chunks...")
    all_factors = pd.concat(all_chunks)
    
    # Save full factors
    factors_path = output_dir / f"factors_{key}.parquet"
    all_factors.to_parquet(factors_path)
    print(f"  ✅ Factors saved: {factors_path}")
    print(f"     Shape: {all_factors.shape}")
    print(f"     Size: {factors_path.stat().st_size / 1024 / 1024 / 1024:.2f} GB")
    
    # Save aligned factors with labels
    if aligned_chunks:
        all_aligned = pd.concat(aligned_chunks)
        aligned_path = output_dir / f"factors_{key}_aligned.parquet"
        all_aligned.to_parquet(aligned_path)
        print(f"  ✅ Aligned data saved: {aligned_path}")
        print(f"     Shape: {all_aligned.shape}")
        print(f"     Size: {aligned_path.stat().st_size / 1024 / 1024 / 1024:.2f} GB")
        
        # Stats
        print(f"  📊 Alignment rate: {len(all_aligned) / len(all_factors) * 100:.2f}%")
        print(f"     NaN in labelValue: {all_aligned['labelValue'].isna().sum():,}")
    
    return all_factors.shape, len(aligned_chunks) > 0

print("Ready to export factor data for keys 0, 1, 2")
print(f"Output directory: {OUTPUT_DIR.absolute()}")
print("\n⚠️ This will take some time and use significant memory!")

In [ ]:
# Export Factor Key 0
export_factor_data(FACTOR_FILE, key="0", labels_indexed=labels_indexed, output_dir=OUTPUT_DIR)

In [ ]:
# Export Factor Key 1
export_factor_data(FACTOR_FILE, key="1", labels_indexed=labels_indexed, output_dir=OUTPUT_DIR)

In [ ]:
# Export Factor Key 2
export_factor_data(FACTOR_FILE, key="2", labels_indexed=labels_indexed, output_dir=OUTPUT_DIR)

In [ ]:
# Summary: List all exported files

print("=" * 60)
print("EXPORT COMPLETE")
print("=" * 60)

print(f"\n📁 Output Directory: {OUTPUT_DIR.absolute()}\n")

total_size = 0
for f in sorted(OUTPUT_DIR.glob("*.parquet")):
    size_mb = f.stat().st_size / 1024 / 1024
    total_size += size_mb
    print(f"  {f.name:35s} {size_mb:>10.2f} MB")

print(f"\n  {'Total':35s} {total_size:>10.2f} MB ({total_size/1024:.2f} GB)")

print("""
📋 File Descriptions:
  - factors_X.parquet:         All factor values for key X (full data)
  - factors_X_aligned.parquet: Factors + labelValue for training
  - labels_aligned.parquet:    All valid labels (endDate <= 20211231)

🔧 Usage Example:
  import pandas as pd
  
  # Load aligned training data
  df = pd.read_parquet('processed_data/factors_0_aligned.parquet')
  X = df.drop(['labelValue', 'endDate'], axis=1)
  y = df['labelValue']
""")